##### Copyright 2024 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# CodeGemma - Common use cases
This notebook demonstrates the basic task that Gemma can solve by using the right prompting.
<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/CodeGemma/[CodeGemma_1]Common_use_cases.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
</table>

## Setup

### Select the Colab runtime
To complete this tutorial, you'll need to have a Colab runtime with sufficient resources to run the Gemma model. In this case, you can use a T4 GPU:

1. In the upper-right of the Colab window, select **▾ (Additional connection options)**.
2. Select **Change runtime type**.
3. Under **Hardware accelerator**, select **T4 GPU**.

### Gemma setup

To complete this tutorial, you'll first need to complete the setup instructions at [Gemma setup](https://ai.google.dev/gemma/docs/setup). The Gemma setup instructions show you how to do the following:

* Get access to Gemma on kaggle.com.
* Select a Colab runtime with sufficient resources to run
  the Gemma 2B model.
* Generate and configure a Kaggle username and an API key as Colab secrets.

After you've completed the Gemma setup, move on to the next section, where you'll set environment variables for your Colab environment.


### Configure your credentials

Add your Kaggle credentials to the Colab Secrets manager to securely store it.

1. Open your Google Colab notebook and click on the 🔑 Secrets tab in the left panel. <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
2. Create new secrets: `KAGGLE_USERNAME` and `KAGGLE_KEY`
3. Copy/paste your username into `KAGGLE_USERNAME`
3. Copy/paste your key into `KAGGLE_KEY`
4. Toggle the buttons on the left to allow notebook access to the secrets.


### Install dependencies
Run the cell below to install all the required dependencies.

In [1]:
!pip install -q -U keras keras-nlp

## Exploring prompting capabilities

### CodeGemma

CodeGemma models are text-to-text and text-to-code decoder-only models that specializes in code completion and code generation tasks. The CodeGemma 2B and 7B variants are specially tuned for code infilling tasks.

This example uses CodeGemma's fill-in-the-middle (FIM) capability to complete code based on the surrounding context. This is particularly useful in code editor applications for inserting code where the text cursor is based on the code around it (before and after the cursor).

CodeGemma lets you use 4 user-defined tokens - 3 (`<|fim_prefix|>`, `<|fim_suffix|>`, `<|fim_middle|>`) for FIM and a `<|file_separator|>` token for multi-file context support. Use these to define constants. More about that in the next cells.


In [ ]:
import os
import keras
import keras_nlp
from dotenv import load_dotenv

load_dotenv()


keras.config.set_floatx("bfloat16")
os.environ["KAGGLE_USERNAME"] = os.getenv("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = os.getenv("KAGGLE_KEY")
os.environ["CUDA_VISIBLE_DEVICES"] = os.getenv("CUDA_VISIBLE_DEVICES")


2025-07-01 21:44:12.266822: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751402652.323042    2930 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751402652.339786    2930 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1751402652.456450    2930 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1751402652.456467    2930 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1751402652.456469    2930 computation_placer.cc:177] computation placer alr

In [3]:
# Load CodeGemma
codegemma = keras_nlp.models.GemmaCausalLM.from_preset("code_gemma_1.1_2b_en")

I0000 00:00:1751402659.316770    2930 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14124 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Ti, pci bus id: 0000:04:00.0, compute capability: 8.9


100%|██████████| 4.67G/4.67G [13:06<00:00, 6.37MB/s]


100%|██████████| 591/591 [00:00<00:00, 268kB/s]


100%|██████████| 4.04M/4.04M [00:01<00:00, 4.05MB/s]
normalizer.cc(51) LOG(INFO) precompiled_charsmap is empty. use identity normalization.


In [4]:
# Tokens
BEFORE_CURSOR = "<|fim_prefix|>"
AFTER_CURSOR = "<|fim_suffix|>"
AT_CURSOR = "<|fim_middle|>"
FILE_SEPARATOR = "<|file_separator|>"
END_TOKEN = codegemma.preprocessor.tokenizer.end_token
stop_tokens = (BEFORE_CURSOR, AFTER_CURSOR, AT_CURSOR, FILE_SEPARATOR, END_TOKEN)

In [5]:
stop_token_ids = tuple(
    codegemma.preprocessor.tokenizer.token_to_id(x) for x in stop_tokens
)

#### Prompting example: Code infilling

In [7]:
import re


# Helpers
def split_response_by_token(response):
    mapping = {}
    parts = re.split(r"(<\|[^\|\>]+\|\>)", response)
    parts = [item for item in parts if len(item)]
    for token in stop_tokens[:3]:
        mapping[token] = ""

        try:
            idx = parts.index(token)
            if parts[idx + 1] not in stop_tokens:
                mapping[token] = parts[idx + 1]
        except (ValueError, IndexError):
            pass

    return mapping

In [8]:
prefix = "def calculate_area_of_rectangle(a: int, b: int) -> int:"
suffix = "\n    return area"
prompt = f"<|fim_prefix|>{prefix}<|fim_suffix|>{suffix}<|fim_middle|>"

response = codegemma.generate(prompt, stop_token_ids=stop_token_ids)
parts = split_response_by_token(response)

print("--- RAW Response ---")
print(response)

print("\n--- The generated (FIM) piece of code: ---")
print(parts[AT_CURSOR])

print("\n--- The whole function: ---")
print(parts[BEFORE_CURSOR], parts[AT_CURSOR], parts[AFTER_CURSOR])

I0000 00:00:1751403529.686598    2930 service.cc:152] XLA service 0x45313060 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1751403529.686616    2930 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Ti, Compute Capability 8.9
2025-07-01 21:58:50.627602: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1751403531.830147    2930 cuda_dnn.cc:529] Loaded cuDNN version 90501
2025-07-01 21:58:54.888515: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_573', 4 bytes spill stores, 4 bytes spill loads

2025-07-01 21:58:55.257653: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_539', 12 

--- RAW Response ---
<|fim_prefix|>def calculate_area_of_rectangle(a: int, b: int) -> int:<|fim_suffix|>
    return area<|fim_middle|>
    """
    Calculate the area of a rectangle.

    Args:
        a (int): The length of the rectangle.
        b (int): The width of the rectangle.

    Returns:
        int: The area of the rectangle.
    """
    area = a * b<|file_separator|>

--- The generated (FIM) piece of code: ---

    """
    Calculate the area of a rectangle.

    Args:
        a (int): The length of the rectangle.
        b (int): The width of the rectangle.

    Returns:
        int: The area of the rectangle.
    """
    area = a * b

--- The whole function: ---
def calculate_area_of_rectangle(a: int, b: int) -> int: 
    """
    Calculate the area of a rectangle.

    Args:
        a (int): The length of the rectangle.
        b (int): The width of the rectangle.

    Returns:
        int: The area of the rectangle.
    """
    area = a * b 
    return area


#### Prompting example: Code Generation
_Note: While the 2B version of CodeGemma was primarily designed for code completion, it can also handle basic code generation tasks. However, for optimal code generation results, we recommend using the instruction-tuned 7B model._

In [9]:
prompt = """Write a one-liner in Python that check if a year is a leap year.
Examples:
>>> is_a_leap_year(2016)
True
>>> is_a_leap_year(2001)
False
>>> is_a_leap_year(2052)
True
def is_a_leap_year(year: int) -> bool:"""

response = codegemma.generate(prompt, max_length=128)
print(response)

2025-07-01 22:00:45.308103: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_573_0', 224 bytes spill stores, 696 bytes spill loads

2025-07-01 22:00:45.777005: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_573', 4 bytes spill stores, 4 bytes spill loads



Write a one-liner in Python that check if a year is a leap year.
Examples:
>>> is_a_leap_year(2016)
True
>>> is_a_leap_year(2001)
False
>>> is_a_leap_year(2052)
True
def is_a_leap_year(year: int) -> bool:
    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)
<|file_separator|>


In [10]:
del codegemma

In [11]:
keras.backend.clear_session()

In [12]:
import gc

gc.collect()

0

#### Re-implement this using purely huggingface models
The current implementation uses Keras which is a specific deep learning framework. Make a new implementation of this notebook that uses Huggingface which is a layer of abstraction above the deep learning framework